[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/43.%20The%20Security%20Inspection%20Optimization%20Problem/P43-Tier-7_executed.ipynb)

---

# P43-Tier-7: Human–AI Partnership for Security Inspection

## Purpose and scope

This notebook simulates a human screener and an AI model assessing the same security items. Their risk estimates are combined using confidence-weighted contributions, and the AI's influence changes as a trust score is updated from observed prediction errors. The simulation also tracks decision categories, fatigue, and performance over a generated set of items.

The method is a sequential simulation, not an optimization algorithm: it does not search for an optimal inspection schedule or guarantee that collaboration outperforms either partner. Its outputs depend on synthetic data and random prediction noise.

## Inputs and outputs

- **Inputs:** synthetic item risk scores and types; model and screener parameters; initial trust; and fixed random seeds.
- **Outputs:** item-level risk estimates and decisions, within-tolerance performance rates, trust history, and descriptive plots and summaries.
- **Evaluation target:** each item's generated risk score. A prediction is counted as correct when its absolute error is less than 0.2. This is a risk-estimation measure, not threat-classification accuracy.

## Modeling assumptions and limitations

The generated risk score is treated as the reference value available to the simulation. The `actual_threat` field is derived from a risk threshold, but it is not used to evaluate the risk estimates. AI and human accuracy, processing speed, the AI confidence threshold, and learning rate are descriptive parameters; the core prediction and trust-update rules do not use them. The `features` field is generated but is not used by either assessor. Trust changes in response to relative prediction accuracy, but neither assessor is retrained. Consequently, the simulation illustrates a particular collaboration rule rather than a validated operational security system.

## Method overview

For each item, the AI and human produce risk estimates and confidence scores. Trust and confidence determine the weights used to combine those estimates. The combined risk maps to one of three actions, and trust is then adjusted according to which partner's risk estimate is within the evaluation tolerance. This sequence is repeated item by item; fatigue is increased at specified intervals during the main simulation.

## Mathematical model

### Confidence-weighted risk estimate

Let \(t \in [0.1,0.9]\) be the trust score before an item is assessed. Let \(r_{AI}\) and \(r_H\) be the AI and human risk estimates, and let \(c_{AI}\) and \(c_H\) be their confidence scores. The model assigns the weights

$$
w_{AI} = t c_{AI}, \qquad w_H = (1-t)c_H.
$$

When the total weight is positive, the collaborative risk estimate is

$$
r_C = \frac{w_{AI}r_{AI}+w_Hr_H}{w_{AI}+w_H},
\qquad
c_C = \frac{w_{AI}+w_H}{2}.
$$

Here, \(r_C\) is the collaborative risk estimate and \(c_C\) is the model's heuristic collaborative-confidence score. The latter is not a calibrated probability. If both weights are zero, the implementation uses the arithmetic mean of the two risk estimates and assigns collaborative confidence 0.5. In the code, these quantities are computed in `HumanAIPartnership.collaborative_decision`.

The action is selected using fixed risk thresholds:

$$
a(r_C)=
\begin{cases}
\text{inspect}, & r_C > 0.7,\\
\text{flag}, & 0.4 < r_C \leq 0.7,\\
\text{clear}, & r_C \leq 0.4.
\end{cases}
$$

These categories are the simulation's decision labels; they are not themselves the performance metric.

### Trust update

Let \(r\) be the reference risk score for the item. Define indicators for estimates within the tolerance \(\epsilon=0.2\):

$$
I_{AI}=\mathbf{1}\{|r_{AI}-r|<\epsilon\},
\qquad
I_H=\mathbf{1}\{|r_H-r|<\epsilon\}.
$$

The trust update is

$$
t' = \operatorname{clip}_{[0.1,0.9]}\left(
\begin{cases}
t+0.05, & I_{AI}=1,\ I_H=0,\\
t-0.05, & I_{AI}=0,\ I_H=1,\\
t, & I_{AI}=I_H.
\end{cases}
\right)
$$

Thus, trust changes only when exactly one assessor is within tolerance; it increases when the AI alone is within tolerance and decreases when the human alone is. The clipping operation keeps trust within the stated bounds. The implementation is `HumanAIPartnership.update_trust`.

### Evaluation and computational scale

For a set of \(n\) items, the within-tolerance rate for an assessor is the number of estimates satisfying \(|\hat r_i-r_i|<0.2\), divided by \(n\). This rate is reported as an accuracy-like summary, but it measures risk-estimation tolerance rather than classification accuracy. Processing the items takes \(O(n)\) prediction and update steps. Retaining item-level decisions and trust history requires \(O(n)\) memory.

## 1. Imports and reproducibility

The random seeds make the synthetic data and stochastic predictions reproducible when the notebook is run from the top with the same library versions. The measured processing time can still vary between runs and machines.

In [ ]:
from dataclasses import dataclass, field
import random
import time
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


# Fixed seeds let the simulation be reproduced from a fresh kernel.
NUMPY_SEED = 43
PYTHON_SEED = 43
np.random.seed(NUMPY_SEED)
random.seed(PYTHON_SEED)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Packages imported successfully.")
print(f"Random seeds set: NumPy={NUMPY_SEED}, Python={PYTHON_SEED}")

## 2. Representing items, assessors, and the partnership

A **state** here means the information retained by the simulation at a particular point: the current trust score, the assessor parameters, and the accumulated records. It is needed because the decision for each item uses the current trust score, while the trust update changes the score used for later items. The dataclasses below make these inputs and evolving records explicit.

The AI and human estimates are synthetic. Each begins with the item's generated risk score, adds random noise, and may receive an expertise adjustment. The noise and adjustments are part of this illustrative model, not empirically calibrated behavior.

In [ ]:
@dataclass
class SecurityItem:
    """An item and its synthetic reference risk for the simulation."""

    id: str
    type: str
    risk_score: float
    features: Dict[str, float] = field(default_factory=dict)
    actual_threat: bool = False
    inspection_time: float = 0.0


@dataclass
class AIModel:
    """Synthetic AI assessor and its descriptive configuration."""

    name: str
    accuracy: float
    processing_speed: float  # Items per minute; descriptive in this simulation.
    confidence_threshold: float
    expertise_areas: List[str] = field(default_factory=list)
    learning_rate: float = 0.1  # Descriptive; the model is not retrained here.

    def predict_risk(self, item: SecurityItem) -> Tuple[float, float]:
        """Return a noisy risk estimate and a simulated confidence score."""
        base_prediction = item.risk_score + np.random.normal(0, 0.1)
        confidence = np.random.beta(5, 2)

        if item.type in self.expertise_areas:
            base_prediction = min(1.0, base_prediction + 0.1)
            confidence = min(1.0, confidence + 0.1)

        return max(0.0, min(1.0, base_prediction)), confidence


@dataclass
class HumanScreener:
    """Synthetic human assessor and fatigue state."""

    name: str
    experience_years: int
    accuracy: float
    inspection_speed: float  # Items per minute; descriptive in this simulation.
    expertise_areas: List[str] = field(default_factory=list)
    fatigue_factor: float = 0.0

    def inspect_item(self, item: SecurityItem) -> Tuple[float, float]:
        """Return a noisy human risk estimate and a fatigue-adjusted confidence."""
        expertise_bonus = 0.15 if item.type in self.expertise_areas else 0.0
        fatigue_penalty = self.fatigue_factor * 0.1

        human_assessment = (
            item.risk_score
            + np.random.normal(0, 0.15)
            + expertise_bonus
            - fatigue_penalty
        )
        confidence = 0.9 - fatigue_penalty

        return (
            max(0.0, min(1.0, human_assessment)),
            max(0.0, min(1.0, confidence)),
        )


@dataclass
class HumanAIPartnership:
    """Combine assessments and update trust from observed prediction errors."""

    ai_model: AIModel
    human_screener: HumanScreener
    trust_score: float = 0.5
    collaboration_history: List[Dict] = field(default_factory=list)
    learning_data: List[Dict] = field(default_factory=list)

    def collaborative_decision(self, item: SecurityItem) -> Dict:
        """Assess an item using trust- and confidence-weighted risk estimates."""
        ai_risk, ai_confidence = self.ai_model.predict_risk(item)
        human_risk, human_confidence = self.human_screener.inspect_item(item)

        ai_weight = self.trust_score * ai_confidence
        human_weight = (1 - self.trust_score) * human_confidence
        total_weight = ai_weight + human_weight

        if total_weight > 0:
            collaborative_risk = (
                ai_weight * ai_risk + human_weight * human_risk
            ) / total_weight
            collaborative_confidence = total_weight / 2
        else:
            collaborative_risk = (ai_risk + human_risk) / 2
            collaborative_confidence = 0.5

        if collaborative_risk > 0.7:
            decision = "inspect"
        elif collaborative_risk > 0.4:
            decision = "flag"
        else:
            decision = "clear"

        return {
            "item_id": item.id,
            "decision": decision,
            "collaborative_risk": collaborative_risk,
            "collaborative_confidence": collaborative_confidence,
            "ai_risk": ai_risk,
            "human_risk": human_risk,
            "ai_confidence": ai_confidence,
            "human_confidence": human_confidence,
            "trust_score": self.trust_score,
        }

    def update_trust(self, outcome: Dict) -> None:
        """Adjust trust when exactly one assessor is within risk tolerance."""
        ai_correct = abs(outcome["ai_risk"] - outcome["actual_risk"]) < 0.2
        human_correct = abs(outcome["human_risk"] - outcome["actual_risk"]) < 0.2

        if ai_correct and not human_correct:
            self.trust_score += 0.05
        elif human_correct and not ai_correct:
            self.trust_score -= 0.05

        self.trust_score = max(0.1, min(0.9, self.trust_score))
        self.learning_data.append(outcome)


print("Human-AI partnership classes are ready.")

## 3. Configure assessors and generate items

The item generator draws reference risks from a beta distribution with parameters \(2\) and \(5\), which places more probability on lower values than on higher ones. It also creates synthetic item types and feature values. The features are retained as part of each item but are not inputs to the assessors in this model.

In [ ]:
def create_ai_model() -> AIModel:
    """Create the AI assessor configuration used in the example."""
    return AIModel(
        name="SecurityAI-v2.0",
        accuracy=0.85,
        processing_speed=60,
        confidence_threshold=0.7,
        expertise_areas=["container", "cargo", "luggage"],
        learning_rate=0.1,
    )


def create_human_screener() -> HumanScreener:
    """Create the human screener configuration used in the example."""
    return HumanScreener(
        name="Senior Agent Smith",
        experience_years=10,
        accuracy=0.88,
        inspection_speed=15,
        expertise_areas=["vehicle", "person", "documents"],
        fatigue_factor=0.0,
    )


def generate_security_items(num_items: int = 100) -> List[SecurityItem]:
    """Generate items, reference risks, and unused descriptive features."""
    item_types = ["container", "vehicle", "cargo", "person", "luggage", "documents"]
    items = []

    for i in range(num_items):
        item_type = random.choice(item_types)
        risk_score = np.random.beta(2, 5)
        actual_threat = risk_score > 0.8
        features = {
            "weight": np.random.lognormal(2, 1),
            "density": np.random.uniform(0.1, 2.0),
            "metal_content": np.random.uniform(0, 1),
            "organic_content": np.random.uniform(0, 1),
        }

        items.append(
            SecurityItem(
                id=f"ITEM-{i + 1:04d}",
                type=item_type,
                risk_score=risk_score,
                features=features,
                actual_threat=actual_threat,
                inspection_time=random.uniform(1, 5),
            )
        )

    return items


ai_model = create_ai_model()
human_screener = create_human_screener()
partnership = HumanAIPartnership(ai_model, human_screener)
security_items = generate_security_items(100)

print(
    f"AI model: {ai_model.name}; descriptive accuracy parameter: "
    f"{ai_model.accuracy:.1%}; processing speed parameter: "
    f"{ai_model.processing_speed} items/min"
)
print(
    f"Human screener: {human_screener.name}; "
    f"{human_screener.experience_years} years of experience; "
    f"descriptive accuracy parameter: {human_screener.accuracy:.1%}"
)
print(f"Generated items: {len(security_items)}")
print(f"Initial trust score: {partnership.trust_score:.2f}")

## 4. Run the sequential collaboration simulation

For each item, the partnership first makes a decision using the current trust score. The simulation then compares both individual estimates and the combined estimate with the generated reference risk. Trust is updated after the decision, so that update affects subsequent items, not the current item's decision. The recorded trust history contains the score after each update, while each item record retains the trust score used to make that item's decision.

In [ ]:
def run_human_ai_simulation(
    partnership: HumanAIPartnership,
    items: List[SecurityItem],
) -> Dict:
    """Run the sequential assessment, evaluation, and trust-update process."""
    if not items:
        raise ValueError("The simulation requires at least one security item.")

    start_time = time.time()
    initial_trust = partnership.trust_score
    results = {
        "decisions": [],
        "trust_evolution": [],
        "initial_trust": initial_trust,
        "performance_metrics": {
            "ai_only": {"correct": 0, "total": 0},
            "human_only": {"correct": 0, "total": 0},
            "collaborative": {"correct": 0, "total": 0},
        },
    }

    for i, item in enumerate(items):
        decision_result = partnership.collaborative_decision(item)
        actual_risk = item.risk_score
        decision_result["actual_risk"] = actual_risk

        ai_correct = abs(decision_result["ai_risk"] - actual_risk) < 0.2
        human_correct = abs(decision_result["human_risk"] - actual_risk) < 0.2
        collaborative_correct = (
            abs(decision_result["collaborative_risk"] - actual_risk) < 0.2
        )

        results["performance_metrics"]["ai_only"]["correct"] += int(ai_correct)
        results["performance_metrics"]["ai_only"]["total"] += 1
        results["performance_metrics"]["human_only"]["correct"] += int(human_correct)
        results["performance_metrics"]["human_only"]["total"] += 1
        results["performance_metrics"]["collaborative"]["correct"] += int(
            collaborative_correct
        )
        results["performance_metrics"]["collaborative"]["total"] += 1

        partnership.update_trust(decision_result)
        results["trust_evolution"].append(partnership.trust_score)
        results["decisions"].append(decision_result)

        # Fatigue affects the human assessment after each completed block of items.
        if i % 20 == 0 and i > 0:
            partnership.human_screener.fatigue_factor = min(
                0.3, partnership.human_screener.fatigue_factor + 0.05
            )

    results["processing_time"] = time.time() - start_time
    return results


simulation_results = run_human_ai_simulation(partnership, security_items)

print("Human-AI partnership simulation results")
print(f"Items processed: {len(simulation_results['decisions'])}")
print(f"Measured processing time: {simulation_results['processing_time']:.3f} seconds")
print(f"Initial trust score: {simulation_results['initial_trust']:.3f}")
print(f"Final trust score: {partnership.trust_score:.3f}")

ai_accuracy = (
    simulation_results["performance_metrics"]["ai_only"]["correct"]
    / simulation_results["performance_metrics"]["ai_only"]["total"]
)
human_accuracy = (
    simulation_results["performance_metrics"]["human_only"]["correct"]
    / simulation_results["performance_metrics"]["human_only"]["total"]
)
collaborative_accuracy = (
    simulation_results["performance_metrics"]["collaborative"]["correct"]
    / simulation_results["performance_metrics"]["collaborative"]["total"]
)

print("\nWithin-tolerance risk-estimation rates")
print(f"AI only: {ai_accuracy:.3f}")
print(f"Human only: {human_accuracy:.3f}")
print(f"Collaborative: {collaborative_accuracy:.3f}")

if ai_accuracy > 0:
    print(
        "Collaborative relative change versus AI: "
        f"{(collaborative_accuracy - ai_accuracy) / ai_accuracy:.1%}"
    )
else:
    print("Collaborative relative change versus AI: undefined because AI rate is zero.")

if human_accuracy > 0:
    print(
        "Collaborative relative change versus human: "
        f"{(collaborative_accuracy - human_accuracy) / human_accuracy:.1%}"
    )
else:
    print(
        "Collaborative relative change versus human: "
        "undefined because human rate is zero."
    )

## 5. Visualize simulation behavior

The panels show trust after each item, within-tolerance rates, decision-label frequencies, confidence-score distributions, risk estimates against reference risks, and rolling absolute prediction errors. A rolling error curve is descriptive: it can rise or fall because of item mix, random noise, trust changes, or fatigue. It does not by itself establish that an assessor learned.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Human-AI Partnership: Simulation Analysis", fontsize=16, fontweight="bold")

# Trust history records the post-update score for each processed item.
trust_scores = simulation_results["trust_evolution"]
items_processed = range(1, len(trust_scores) + 1)
axes[0, 0].plot(items_processed, trust_scores, color="blue", linewidth=2, label="Trust after item")
axes[0, 0].axhline(
    y=simulation_results["initial_trust"],
    color="gray",
    linestyle="--",
    alpha=0.7,
    label="Initial trust",
)
axes[0, 0].set_xlabel("Items processed")
axes[0, 0].set_ylabel("Trust score")
axes[0, 0].set_title("Trust Evolution")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

methods = ["AI only", "Human only", "Collaborative"]
accuracies = [ai_accuracy, human_accuracy, collaborative_accuracy]
colors = ["red", "blue", "green"]
bars = axes[0, 1].bar(methods, accuracies, color=colors, alpha=0.7, edgecolor="black")
axes[0, 1].set_ylabel("Within-tolerance rate")
axes[0, 1].set_title("Risk-Estimation Performance")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(True, axis="y", alpha=0.3)
for bar, accuracy in zip(bars, accuracies):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{accuracy:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )

# All three decision labels are displayed in a stable order, including zero counts.
decision_labels = ["clear", "flag", "inspect"]
decision_counts = {
    label: sum(
        decision["decision"] == label
        for decision in simulation_results["decisions"]
    )
    for label in decision_labels
}
nonzero_decision_counts = {
    label: count for label, count in decision_counts.items() if count > 0
}
axes[0, 2].pie(
    nonzero_decision_counts.values(),
    labels=nonzero_decision_counts.keys(),
    autopct="%1.1f%%",
    startangle=90,
)
axes[0, 2].set_title("Collaborative Decision Labels")

ai_confidences = [d["ai_confidence"] for d in simulation_results["decisions"]]
human_confidences = [d["human_confidence"] for d in simulation_results["decisions"]]
collab_confidences = [
    d["collaborative_confidence"] for d in simulation_results["decisions"]
]
axes[1, 0].hist(ai_confidences, bins=20, alpha=0.5, label="AI", color="red")
axes[1, 0].hist(human_confidences, bins=20, alpha=0.5, label="Human", color="blue")
axes[1, 0].hist(
    collab_confidences,
    bins=20,
    alpha=0.5,
    label="Collaborative heuristic",
    color="green",
)
axes[1, 0].set_xlabel("Confidence score")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("Confidence-Score Distributions")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

ai_risks = [d["ai_risk"] for d in simulation_results["decisions"]]
human_risks = [d["human_risk"] for d in simulation_results["decisions"]]
collab_risks = [d["collaborative_risk"] for d in simulation_results["decisions"]]
actual_risks = [d["actual_risk"] for d in simulation_results["decisions"]]
axes[1, 1].scatter(actual_risks, ai_risks, alpha=0.5, label="AI", color="red")
axes[1, 1].scatter(actual_risks, human_risks, alpha=0.5, label="Human", color="blue")
axes[1, 1].scatter(
    actual_risks, collab_risks, alpha=0.5, label="Collaborative", color="green"
)
axes[1, 1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Exact agreement")
axes[1, 1].set_xlabel("Reference risk score")
axes[1, 1].set_ylabel("Estimated risk score")
axes[1, 1].set_title("Risk Estimates versus Reference")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

window_size = 10
ai_errors = []
human_errors = []
collab_errors = []
for i in range(window_size, len(simulation_results["decisions"]) + 1):
    window_decisions = simulation_results["decisions"][i - window_size : i]
    ai_errors.append(
        np.mean(
            [abs(d["ai_risk"] - d["actual_risk"]) for d in window_decisions]
        )
    )
    human_errors.append(
        np.mean(
            [abs(d["human_risk"] - d["actual_risk"]) for d in window_decisions]
        )
    )
    collab_errors.append(
        np.mean(
            [
                abs(d["collaborative_risk"] - d["actual_risk"])
                for d in window_decisions
            ]
        )
    )

error_item_numbers = range(window_size, len(simulation_results["decisions"]) + 1)
axes[1, 2].plot(error_item_numbers, ai_errors, "r-", label="AI", alpha=0.7)
axes[1, 2].plot(error_item_numbers, human_errors, "b-", label="Human", alpha=0.7)
axes[1, 2].plot(
    error_item_numbers,
    collab_errors,
    "g-",
    label="Collaborative",
    alpha=0.8,
    linewidth=2,
)
axes[1, 2].set_xlabel("Items processed")
axes[1, 2].set_ylabel(f"Mean absolute error (window={window_size})")
axes[1, 2].set_title("Rolling Prediction Error")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Analyze trust, contribution weights, and error change

A decision is classified as AI-dominant or human-dominant when one assessor's trust-confidence weight exceeds the other's by a factor of 1.5. Otherwise, it is classified as balanced. These labels describe relative weights in the combination rule; they do not measure the quality of the final decision.

The relative error reduction compares the mean collaborative absolute error in the first half with that in the second half. A positive value means the second-half error is lower; a negative value means it is higher. This comparison is descriptive and is not proof of learning or causation.

In [ ]:
def analyze_partnership_dynamics(results: Dict) -> Dict:
    """Summarize trust changes, relative weights, and error across the run."""
    decisions = results["decisions"]
    trust_evolution = results["trust_evolution"]

    if not decisions or not trust_evolution:
        raise ValueError("Partnership analysis requires a non-empty simulation.")

    trust_change = trust_evolution[-1] - results["initial_trust"]
    trust_volatility = float(np.std(trust_evolution))

    ai_dominant_decisions = 0
    human_dominant_decisions = 0
    balanced_decisions = 0

    for decision in decisions:
        ai_weight = decision["trust_score"] * decision["ai_confidence"]
        human_weight = (1 - decision["trust_score"]) * decision["human_confidence"]

        if ai_weight > human_weight * 1.5:
            ai_dominant_decisions += 1
        elif human_weight > ai_weight * 1.5:
            human_dominant_decisions += 1
        else:
            balanced_decisions += 1

    midpoint = len(decisions) // 2
    first_half = decisions[:midpoint]
    second_half = decisions[midpoint:]

    first_half_error = float(
        np.mean(
            [abs(d["collaborative_risk"] - d["actual_risk"]) for d in first_half]
        )
    )
    second_half_error = float(
        np.mean(
            [abs(d["collaborative_risk"] - d["actual_risk"]) for d in second_half]
        )
    )
    relative_error_reduction = (
        (first_half_error - second_half_error) / first_half_error
        if first_half_error > 0
        else float("nan")
    )

    total_decisions = len(decisions)
    return {
        "trust_change": trust_change,
        "trust_volatility": trust_volatility,
        "final_trust": trust_evolution[-1],
        "ai_dominant_ratio": ai_dominant_decisions / total_decisions,
        "human_dominant_ratio": human_dominant_decisions / total_decisions,
        "balanced_ratio": balanced_decisions / total_decisions,
        "relative_error_reduction": relative_error_reduction,
        "first_half_error": first_half_error,
        "second_half_error": second_half_error,
    }


dynamics = analyze_partnership_dynamics(simulation_results)

print("Partnership dynamics")
print(f"Trust change from initial value: {dynamics['trust_change']:+.3f}")
print(f"Trust-score standard deviation: {dynamics['trust_volatility']:.3f}")
print(f"Final trust score: {dynamics['final_trust']:.3f}")
print("\nRelative contribution-weight categories")
print(f"AI-dominant: {dynamics['ai_dominant_ratio']:.1%}")
print(f"Human-dominant: {dynamics['human_dominant_ratio']:.1%}")
print(f"Balanced: {dynamics['balanced_ratio']:.1%}")
print("\nCollaborative absolute error by half of the run")
if np.isfinite(dynamics["relative_error_reduction"]):
    print(
        "Relative error reduction (positive means lower second-half error): "
        f"{dynamics['relative_error_reduction']:.1%}"
    )
else:
    print("Relative error reduction is undefined because first-half error is zero.")
print(f"First-half mean absolute error: {dynamics['first_half_error']:.3f}")
print(f"Second-half mean absolute error: {dynamics['second_half_error']:.3f}")

## 7. Compare expertise settings by item type

This experiment evaluates the same collaboration rule on items grouped by type, using fresh assessors configured with expertise in the tested type. Each group uses up to 20 items; when the main generated set contains fewer than five items of a type, additional synthetic items are generated for that group. Results are stochastic and are based on the same within-tolerance definition as the main simulation. This is a descriptive comparison, not a guarantee that expertise causes a particular performance level.

In [ ]:
def simulate_expertise_synergy(
    security_items: List[SecurityItem],
    ai_model: AIModel,
    human_screener: HumanScreener,
) -> Dict:
    """Compare the collaboration rule across item types with fresh assessors."""
    item_types = ["container", "vehicle", "cargo", "person", "luggage", "documents"]
    synergy_results = {}

    for item_type in item_types:
        type_items = [item for item in security_items if item.type == item_type]

        if len(type_items) < 5:
            for i in range(20):
                risk_score = np.random.beta(2, 5)
                type_items.append(
                    SecurityItem(
                        id=f"{item_type.upper()}-SUPPLEMENT-{i + 1:02d}",
                        type=item_type,
                        risk_score=risk_score,
                        actual_threat=risk_score > 0.8,
                    )
                )

        type_partnership = HumanAIPartnership(
            AIModel(
                name=ai_model.name,
                accuracy=ai_model.accuracy,
                processing_speed=ai_model.processing_speed,
                confidence_threshold=ai_model.confidence_threshold,
                expertise_areas=[item_type],
                learning_rate=ai_model.learning_rate,
            ),
            HumanScreener(
                name="Expert Agent",
                experience_years=human_screener.experience_years,
                accuracy=human_screener.accuracy,
                inspection_speed=human_screener.inspection_speed,
                expertise_areas=[item_type],
            ),
        )

        tested_items = type_items[:20]
        type_results = run_human_ai_simulation(type_partnership, tested_items)
        performance = type_results["performance_metrics"]["collaborative"]
        type_accuracy = performance["correct"] / performance["total"]

        synergy_results[item_type] = {
            "accuracy": type_accuracy,
            "items_tested": len(tested_items),
            "ai_expertise": item_type in ai_model.expertise_areas,
            "human_expertise": item_type in human_screener.expertise_areas,
        }

    return synergy_results


synergy_results = simulate_expertise_synergy(
    security_items,
    ai_model,
    human_screener,
)

print("Expertise-setting comparison")
for item_type, results in synergy_results.items():
    ai_expertise = "yes" if results["ai_expertise"] else "no"
    human_expertise = "yes" if results["human_expertise"] else "no"
    print(
        f"{item_type.capitalize()}: {results['accuracy']:.3f} within-tolerance rate; "
        f"{results['items_tested']} items; AI expertise: {ai_expertise}; "
        f"human expertise: {human_expertise}"
    )

best_accuracy = max(result["accuracy"] for result in synergy_results.values())
best_types = [
    item_type
    for item_type, result in synergy_results.items()
    if np.isclose(result["accuracy"], best_accuracy, rtol=1e-9, atol=1e-12)
]
print(
    "Highest observed within-tolerance rate in this run: "
    f"{', '.join(best_types)} ({best_accuracy:.3f})"
)

## 8. Interpret the results

The reported rates and trust trajectory describe this seeded synthetic run. A higher collaborative within-tolerance rate than either individual rate would indicate better performance under this simulation's specific data, noise, thresholds, and update rule; it would not establish general superiority. A rising trust score means the AI alone met the chosen tolerance more often than the human alone on trust-update comparisons, not that the AI is universally more reliable. The expertise experiment likewise compares simulated settings rather than validated real-world capabilities.

The method's defining sequence remains explicit: generate an item, obtain both estimates, combine them using the current trust and confidence values, assign a risk-based label, evaluate against the reference risk, and update trust for the next item. Further analysis can examine additional item sets or additional operational constraints without changing that sequence.

In [ ]:
processing_rate = (
    len(security_items) / simulation_results["processing_time"]
    if simulation_results["processing_time"] > 0
    else float("inf")
)

print("=" * 72)
print("P43-Tier-7: Human-AI Partnership Simulation Summary")
print("=" * 72)
print(f"Items processed: {len(security_items)}")
print(f"Initial trust score: {simulation_results['initial_trust']:.3f}")
print(f"Final trust score: {dynamics['final_trust']:.3f}")
print(f"Trust change: {dynamics['trust_change']:+.3f}")
print("\nWithin-tolerance risk-estimation rates")
print(f"AI only: {ai_accuracy:.3f}")
print(f"Human only: {human_accuracy:.3f}")
print(f"Collaborative: {collaborative_accuracy:.3f}")
print("\nRelative contribution-weight categories")
print(f"AI-dominant: {dynamics['ai_dominant_ratio']:.1%}")
print(f"Human-dominant: {dynamics['human_dominant_ratio']:.1%}")
print(f"Balanced: {dynamics['balanced_ratio']:.1%}")
print("\nDescriptive execution measurements")
print(f"Measured processing time: {simulation_results['processing_time']:.3f} seconds")
print(f"Measured throughput: {processing_rate:.0f} items/second")
print("\nInterpret these figures as results of a synthetic simulation, not as")
print("validated estimates of real-world inspection performance.")
print("=" * 72)